# Trajectories of magnetic soft robot

### Imports

In [34]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, FlowBodySolver, shear_flow, oscillating_magnetic_field
from softmobility.classes.flowbodysolver import _rotation_matrix_from_Rodrigues_impl
from softmobility.classes.flowbodysolver import compute_Bortz_operator, rescale_orientation

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

### Magnetic parameters

In [35]:
MAGNETIC_FORCING = 10   # Magnetic amplitude: m Bx / (mu a omega)
MAGNETIC_RATIO = 1      # Magnetic ratio: By /Bx

## Simulation of default parameters

### YAML file

In [36]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k    
  - radius
  - length
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  _ref_: 1
  spring_k1: .5           
  spring_k2: .5           
  radius1: .5
  radius2: .5
  length1: .5
  length2: .5
  _spr_: 100
  _rad_: 2
  _len_: 5
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: _ref_                    
    orientation: [0, 0, phi1]
    torque: 
      - 0
      - 0
      - "magnetic1 * cos(phi1) - magnetic0 * sin(phi1) - _spr_ * spring_k1 * phi1"              

  - # Sphere 1 #################
    radius: radius1               
    position: [_ref_ + _rad_ * radius1 + _len_ * length1, 0, 0]       
    # torque: [0, 0, _spr_ * spring_k1 * phi1]
    torque: [0, 0, _spr_ * (spring_k1 * phi1 +  spring_k2 * phi2)]

  - # Sphere 2 #################
    radius: radius2               
    position: 
      - "_ref_ + _rad_ * radius1 + _len_ * length1 + (_rad_ * (radius1 + radius2) + _len_ * length2) * cos(phi2)"
      - "(_rad_ * (radius1 + radius2) + _len_ * length2) *  sin(phi2)"
      - 0
    orientation: [0, 0, phi2]       
    torque: [0, 0, -_spr_ * spring_k2 * phi2]      
"""

In [37]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k    
  - radius
  - base
  - arm
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  _ref_: 1
  spring_k0: .5
  spring_k1: .5           
  radius1: .5
  base1: .5
  arm1: .5
  _spr_: 100
  _rad_: 2
  _len_: 10
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: _ref_                    
    orientation: [0, 0, phi0]
    torque: 
      - 0
      - 0
      - "magnetic1 * cos(phi0) - magnetic0 * sin(phi0) - _spr_ * spring_k0 * phi0"              
      
  - # Sphere 1 #################
    radius: _rad_ * radius1               
    position: [ _len_ * (base1 - 0.5) + _len_ * arm1 * cos(-phi0 * spring_k0 / spring_k1), _len_ * arm1 * sin(-phi0 * spring_k0 / spring_k1), 0]  
    orientation: [0, 0, -phi0 * spring_k0 / spring_k1]     
    torque: [0, 0, _spr_ * spring_k0 * phi0]   
"""

### Soft Body

In [38]:
mybody= SoftBody(yaml_data, verbose=True)

  Found variables: phi0, arm1, base1, radius1, spring_k0, spring_k1, magnetic0, magnetic1, magnetic2
  3D field inputs:  ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'phi0']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['-sin(phi0)', 'cos(phi0)', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['-100*spring_k0']]
    Sphere 1
      Radius: 2*radius1
      Position: ['10*arm1*cos(phi0*spring_k0/spring_k1) + 10*base1 - 5.0', '-10*arm1*sin(phi0*spring_k0/spring_k1)', '0']
      Orientation: ['0', '0', '-phi0*spring_k0/spring_k1']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['100*spring_k0']]


### Flow-body solver

In [39]:
# Create a shear flow with shear rate 1
no_flow = shear_flow(shear_rate=0)
# Create a magnetic field 
MAGNETIC_FIELD = oscillating_magnetic_field(amp_x=MAGNETIC_FORCING, amp_y=MAGNETIC_RATIO * MAGNETIC_FORCING, omega=1)

print(MAGNETIC_FIELD.vector(time=0))
print(MAGNETIC_FIELD.vector(time=0.2))

[ 10.   0.   0.]
[ 10.      1.987   0.   ]


In [40]:
# parameters
final_time = 12 * 2 * jnp.pi  
DT = 0.1

# optimal design parameters
# mybody.set_design_defaults(new_dict={
#     'length1': 0.715,
#     'length2': 0.788,
#     'radius1': 0.001,
#     'radius2': 0.644,
#     'spring_k1': 0.673, 
#     'spring_k2': 0.001, 
#     })

solver = FlowBodySolver(
    soft_body=mybody, 
    flow=no_flow, 
    input_map={"magnetic": MAGNETIC_FIELD}, 
    dt=DT, 
    init_orientation=[0, 0, 0], 
    integrator="RK2")
trajectory = solver.simulate(final_time=final_time)

Time: 0.000 / 75.398  Integrator RK2
Time: 10.000 / 75.398  Integrator RK2
Time: 20.000 / 75.398  Integrator RK2
Time: 30.000 / 75.398  Integrator RK2
Time: 40.000 / 75.398  Integrator RK2
Time: 50.000 / 75.398  Integrator RK2
Time: 60.000 / 75.398  Integrator RK2
Time: 70.000 / 75.398  Integrator RK2


### Figure of trajectory

In [41]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2][0] for t in trajectory])

# Time vector
time = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=time, y=dofs[:], mode='lines', name='DOF 0'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=time, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=time, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=time, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=time, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()

## Numerical optimization

### Parameters

In [42]:
N_DT = 64               # number of time steps per period
N_TRANSIENT = 10        # number of periods before optimization begins
MAGNETIC_FORCING = 10   # Magnetic amplitude: m Bx / (mu a omega)
MAGNETIC_RATIO = 1      # Magnetic ratio: By /Bx

DT = 2.0 * np.pi / N_DT
MAGNETIC_FIELD = oscillating_magnetic_field(amp_x=MAGNETIC_FORCING, amp_y=MAGNETIC_RATIO * MAGNETIC_FORCING, omega=1)

### Functions

In [43]:
rotation_matrix_from_Rodrigues = jax.jit(
    lambda rvec, Ndof: _rotation_matrix_from_Rodrigues_impl(rvec, Ndof), static_argnums=(1,)
)

In [44]:
def sixc_velocity(design, orientation, dofs, field_lab):
    rot_matrix, sixc_rot_matrix = rotation_matrix_from_Rodrigues(orientation, Ndof=mybody.Ndof)
    field_body = rot_matrix.T @ field_lab
    tensors = mybody.compute_mobility_problem(dofs, design)
    sixc_velocity = tensors.M_H @ field_body + tensors.M_K @ dofs

    return sixc_rot_matrix @ sixc_velocity 

In [45]:
def _rollout_one_period(design, position, orientation, dofs):

    def step(carry, t):
        position, orientation, dofs = carry
        time = t * DT
        current_position = position.copy()
        current_orientation = orientation.copy()
        current_dofs = dofs.copy()

        field_lab = MAGNETIC_FIELD.vector(position, time)
        bortz     = compute_Bortz_operator(orientation)
        p1        = sixc_velocity(design, orientation, dofs, field_lab)
        
        k1_pos = p1[:3]
        k1_ori = p1[3:6]
        k1_dof = p1[6:]

        position    = position    + DT * k1_pos / 2
        orientation = orientation + DT * bortz @ k1_ori / 2
        dofs        = dofs        + DT * k1_dof / 2
        
        field_lab = MAGNETIC_FIELD.vector(position, time)
        p2        = sixc_velocity(design, orientation, dofs, field_lab)  

        k2_pos = p2[:3]
        k2_ori = p2[3:6]
        k2_dof = p2[6:]

        position    = current_position    + DT * (k1_pos + k2_pos) / 2
        orientation = current_orientation + DT * bortz @ (k1_ori + k2_ori) / 2
        orientation = rescale_orientation(orientation)
        dofs        = current_dofs        + DT * (k1_dof + k2_dof) / 2
                
        return (position, orientation, dofs), None

    timesteps = jnp.arange(N_TRANSIENT * N_DT)
    (position, orientation, dofs), _ = jax.lax.scan(
        step, (position, orientation, dofs), timesteps
    )

    x_init    = position[0].copy()   # capture before evaluation
    timesteps = jnp.arange(N_DT)
    (position, orientation, dofs), _ = jax.lax.scan(
        step, (position, orientation, dofs), timesteps
    )
    mean_vx = (position[0] - x_init) / (2. * np.pi)   # displacement from true start

    return position, orientation, dofs, mean_vx

rollout_one_period = jax.jit(_rollout_one_period)

In [46]:
def _mean_vx(design, position, orientation, dofs):
    return rollout_one_period(design, position, orientation, dofs)[-1]

grad_vx_mean = jax.jit(jax.grad(_mean_vx, argnums=0))

In [47]:
def optimize_online(init_design, n_warmup=10, n_periods=200, learning_rate=1e-3, momentum=0.9, maximize=True):
    """
    Gradient ascent on design over multiple periods, with warmup transient.
    """
    design      = jnp.array(init_design)
    position    = jnp.array([0., 0., 0.])
    orientation = jnp.array([0., 0., 0.])
    dofs        = jnp.zeros(mybody.Ndof)
    velocity    = jnp.zeros_like(design)
    if maximize:
        sign = +1
    else:
        sign = -1

    # --- Warmup: run n_warmup periods without gradient update ---
    print("Warming up...")
    for _ in range(n_warmup):
        position, orientation, dofs, mean_vx = rollout_one_period(design, position, orientation, dofs)
        
    print(f"Warmup done — position={position[0]}, mean_vx={float(mean_vx):.6f}")

    # --- Gradient ascent ---
    for period in range(n_periods):
        grad = sign * grad_vx_mean(design, position, orientation, dofs)
                
        if jnp.any(jnp.isnan(grad)):
            print(f"  Warning: NaN gradient at period {period}, skipping update")
        else:
            velocity = momentum * velocity + learning_rate * grad / (jnp.linalg.norm(grad) + 1.E-6)
            design   = jnp.clip(design + velocity, 3 * learning_rate, 1.0)

        # Advance state with updated design as buffer
        position, orientation, dofs, mean_vx = rollout_one_period(design, position, orientation, dofs)
            
        if period % 100 == 0:
            print(f"period {period:4d}  mean_vx={float(mean_vx):.6f}  "
                  f"|grad|={jnp.linalg.norm(grad):.4f}  design={design}")

    return design

### Simulation and optimization

In [48]:
opt_design = optimize_online(
    init_design=mybody.design_defaults,
    n_warmup=N_TRANSIENT,
    n_periods=1001,
    learning_rate=0.001,
    momentum=0.8,
    maximize=False,
)

print("\nOPTIMUM DESIGN PARAMETERS:\n", mybody.design_variables)
print(opt_design)

Warming up...
Warmup done — position=0.022013837471604347, mean_vx=0.000031
period    0  mean_vx=0.000030  |grad|=0.0011  design=[ 0.5    0.501  0.5    0.5    0.5  ]
period  100  mean_vx=-0.000482  |grad|=0.0025  design=[ 0.592  0.839  0.57   0.544  0.243]
period  200  mean_vx=-0.002009  |grad|=0.0080  design=[ 0.637  0.876  0.55   0.567  0.009]
period  300  mean_vx=-0.002184  |grad|=0.0598  design=[ 0.672  0.895  0.5    0.567  0.006]
period  400  mean_vx=-0.002343  |grad|=0.0335  design=[ 0.694  0.918  0.469  0.567  0.008]
period  500  mean_vx=-0.002428  |grad|=0.0200  design=[ 0.712  0.942  0.445  0.567  0.008]
period  600  mean_vx=-0.002486  |grad|=0.0699  design=[ 0.728  0.967  0.424  0.567  0.006]
period  700  mean_vx=-0.002607  |grad|=0.0365  design=[ 0.742  0.992  0.408  0.567  0.008]
period  800  mean_vx=-0.002622  |grad|=0.0208  design=[ 0.754  1.     0.394  0.567  0.009]
period  900  mean_vx=-0.002611  |grad|=0.0672  design=[ 0.765  1.     0.386  0.567  0.006]
period 1000  me

## Optimum gait

In [49]:
mybody.set_design_defaults(new_design=opt_design)

OLD default param values: ['arm1', 'base1', 'radius1', 'spring_k0', 'spring_k1'] [ 0.5  0.5  0.5  0.5  0.5]
NEW default param values: ['arm1', 'base1', 'radius1', 'spring_k0', 'spring_k1'] [ 0.775  1.     0.381  0.567  0.008]


In [50]:
# parameters
final_time = 12 * 2 * jnp.pi  

solver = FlowBodySolver(
    soft_body=mybody, 
    flow=no_flow, 
    input_map={"magnetic": MAGNETIC_FIELD}, 
    dt=DT, 
    init_orientation=[0, 0, 0], 
    integrator="RK2")
trajectory = solver.simulate(final_time=final_time)

Time: 0.000 / 75.398  Integrator RK2
Time: 9.817 / 75.398  Integrator RK2
Time: 19.635 / 75.398  Integrator RK2
Time: 29.452 / 75.398  Integrator RK2
Time: 39.270 / 75.398  Integrator RK2
Time: 49.087 / 75.398  Integrator RK2
Time: 58.905 / 75.398  Integrator RK2
Time: 68.722 / 75.398  Integrator RK2


### Figure of trajectory

In [51]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2][0] for t in trajectory])

# Time vector
time = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=time, y=dofs[:], mode='lines', name='DOF 0'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=time, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=time, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=time, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=time, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()